# Lecture 12 — 风险度量与 VaR（整理版）
按教学逻辑整理，包含单资产VaR、多期VaR、正态性检验、Cornish-Fisher修正、Expected Shortfall、以及投资组合波动率。

## 导入库

In [ ]:
import numpy as np
import pandas as pd
from math import sqrt
from scipy import stats
from scipy.stats import norm
import yfinance as yf

## 1. 单只股票持仓价值（基准）

In [ ]:
# 获取 IBM 股票数据
df = yf.download('IBM')
df = pd.read_pickle('ibm.pkl')

# 计算 100 股 IBM 的持仓价值
nStocks = 100
wealth = nStocks * df['Close'].iloc[-1]
wealth = round(wealth, 2)
print(f'the value of {nStocks} shares of IBM is {wealth}')

## 2. 单日 VaR（参数法 - 正态分布假设）

In [ ]:
# 计算 1000 股 IBM 的单日 VaR
from scipy.stats import norm
ticker = 'IBM'
n_shares = 1000
confidence_level = 0.99
z = norm.ppf(1 - confidence_level)

ret = df['Adj Close'].pct_change().dropna()
position = round(n_shares * df['Adj Close'][-1], 2)
stdev = ret.std()
mean = ret.mean()
var = round(position * (mean - z * stdev), 2)

# 简化公式
var_s = round(position * (-z * stdev), 2)

lastday = df.index[-1].date()
print(f'holding={position}, var={var}')
print(f'simplified var={var_s} tomorrow')
print(f'number of shares={n_shares}, today={lastday}')

## 3. 单日 VaR（历史法 - 排序法）

In [ ]:
# VaR 的排序方法（非参数法）
n_shares = 500
confidence_level = 0.99
df['ret'] = df['Adj Close'].pct_change().dropna()
ret2 = np.sort(df['ret'])
position = round(n_shares * df['Adj Close'][-1], 2)
n = len(ret2)
m = int(n * (1 - confidence_level))

var_sort = round(position * ret2[m], 2)
print(f'var sorted={var_sort} next day, wealth={position}')

## 4. 多日 VaR - 10 天收益（参数法）

In [ ]:
# 计算 10 天收益的 VaR
ticker = 'WMT'
ndays = 10

df = yf.download(ticker)
df = pd.read_pickle('wmt.pkl')

df['retplus'] = df['Adj Close'].pct_change().dropna() + 1
ddate = []
for i in range(0, df.shape[0]):
    ddate.append(int(i / ndays))
df['ndays'] = ddate

ret2 = df.retplus.groupby(df.ndays).prod() - 1

# 基于 10 日收益率计算 VaR
from scipy.stats import norm
n_shares = 50
confidence_level = 0.99
n_days = 10
z = norm.ppf(1 - confidence_level)

df = pd.read_pickle('wmt2.pkl')
df['retplus1'] = df['Adj Close'].pct_change().dropna() + 1
ddate = []
for i in range(0, df.shape[0]):
    ddate.append(int(i / ndays))
df['ndays'] = ddate
ret2 = df.retplus1.groupby(df.ndays).prod() - 1
mean = ret2.mean()
stdev = ret2.std()
position = round(n_shares * df['Adj Close'][0], 2)
var = round(position * (mean - z * stdev), 2)
print(f'number of shares={n_shares} for wmt')
print(f'holding={position}, var={var} in {n_days} days')

## 5. 多日 VaR - 10 天收益（历史法 - 排序法）

In [ ]:
# 10 天 VaR 排序法
ddate = []
for i in range(0, df.shape[0]):
    ddate.append(int(i / ndays))
df['ndays'] = ddate
ret2 = df.retplus1.groupby(df.ndays).prod() - 1
ret3 = np.sort(ret2)
n = len(ret3)
lefttail = int(n * (1 - confidence_level))
var_sort = round(position * ret3[lefttail], 2)

print(f'holding={position}, var={var_sort} in {n_days} days')

## 6. 正态性检验

In [ ]:
# 检验收益是否服从正态分布
from scipy import stats

df = pd.read_pickle('ibm.pkl')
ret = df['Adj Close'].pct_change().dropna()
print(stats.shapiro(ret))
print(stats.anderson(ret))

## 7. 偏度与峰度的计算

In [ ]:
# 计算收益分布的偏度与峰度
from numpy import mean, std, power, random

np.random.seed(12345)
n = 5000000
ret = random.normal(loc=0, scale=1, size=n)
mean_val = ret.mean()
std_val = ret.std()

skew = 0
kurt = 0
for r in ret:
    skew += power((r - mean_val) / std_val, 3)
    kurt += power((r - mean_val) / std_val, 4)

skew /= (std_val ** 3) * (n - 1)
kurt /= (std_val ** 4) * (n - 1)
print(f'skewness={skew}, kurtosis={kurt}')

## 8. Cornish-Fisher 修正的 VaR（基于高阶矩）

In [ ]:
# 基于偏度和峰度的修正 VaR
df = pd.read_pickle('ibm.pkl')
confidence_level = 0.99
n_shares = 500
z = stats.norm.ppf(1 - confidence_level)
ret = df['Adj Close'].pct_change().dropna()
position = round(n_shares * df['Adj Close'][-1], 2)

# 参数法 VaR
var1 = round(position * (ret.mean() - z * ret.std()), 2)

# 修正 VaR - Cornish-Fisher
s = stats.skew(ret)
k = stats.kurtosis(ret)
t = z + 1/6 * (z**2 - 1) * s + 1/24 * (z**3 - 3*z) * k - 1/36 * (2*z**3 - 5*z) * s**2
var2 = round(position * (ret.mean() - t * ret.std()), 2)
print(f'holding={position}, var={var1}, modified_var={var2} next day')

## 9. Expected Shortfall（条件期望损失）

In [ ]:
# Expected Shortfall 的计算
ret2 = np.sort(df['ret'])
position = round(n_shares * df['Adj Close'][-1], 2)
n = ret2.shape[0]
m = int(n * (1 - confidence_level))

sum_val = 0.0
for i in np.arange(m):
    sum_val += ret2[i]
ret_tail = sum_val / m
es = round(position * ret_tail, 2)
print(f'expected shortfall={es}')

## 10. 两资产组合波动率

In [ ]:
# 计算两个资产组合的波动率
def portfolio_volatility(ret1, ret2, w1):
    x1 = w1
    x2 = 1 - w1
    s1 = ret1.std()
    s2 = ret2.std()
    v = x1**2 * s1**2 + x2**2 * s2**2 + 2 * x1 * x2 * np.cov(ret1, ret2)[1][1]
    return sqrt(v)

# 模拟数据示例
from numpy import random
np.random.seed(12345)
n = 50000
m1, m2 = 0.02, 0.05
std1_0, std2_0 = 0.2, 0.12
ret1 = random.normal(loc=m1, scale=std1_0, size=n)
ret2 = random.normal(loc=m2, scale=std2_0, size=n)
w1 = 0.4
std1 = round(ret1.std(), 6)
std2 = round(ret2.std(), 6)
p_vol = round(portfolio_volatility(ret1, ret2, w1), 6)
print(f'vol of a 2-stock port ={p_vol}, w={w1}, std1={std1}, std2={std2}')

## 11. 实际数据：IBM 和 WMT 的两资产组合

In [ ]:
# 实际股票数据的两资产组合波动率
df = pd.read_pickle('ibm.pkl')
df = pd.DataFrame(df)
retibm = df['Adj Close'].pct_change().dropna()
retibm.columns = ['retibm']

df2 = pd.read_pickle('wmt.pkl')
df2 = pd.DataFrame(df2)
retwmt = df2['Adj Close'].pct_change().dropna()
retwmt.columns = ['retwmt']

w1 = 0.5
final = retibm.merge(retwmt, left_index=True, right_index=True)
std1 = round(final.retibm.std(), 6)
std2 = round(final.retwmt.std(), 6)
p_vol = round(portfolio_volatility(final.retibm, final.retwmt, w1), 6)
print(f'vol of a 2-stock port ={p_vol}, w1={w1}')

## 12. N 资产组合波动率

In [ ]:
# N 资产组合波动率的计算
def vol_portfolio(ret_matrix, w):
    m = ret_matrix.shape[1]
    m2 = len(w)
    if m != m2:
        print('the number of stocks and weights do not match')
    else:
        cov = np.cov(ret_matrix.T)
        final = 0
        n = np.arange(0, m)
        for i in n:
            for j in n:
                final += w[i] * w[j] * cov[i][j]
        std_port = sqrt(final)
        return std_port

# 模拟 10 资产组合的波动率
nreturns = 500
nstocks = 10
np.random.seed(12345)
ret_matrix = np.random.uniform(low=-0.12, high=0.05, size=(nreturns, nstocks))

w = np.ones(nstocks) / nstocks
vol = round(vol_portfolio(ret_matrix, w), 6)
print(f'vol of a {nstocks}-stock port ={vol}, w={w}')